In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np

# ------------------- LOAD DATA -------------------
fairways = gpd.read_file("fairways.csv")
greens   = gpd.read_file("greens.csv")
bunkers  = gpd.read_file("bunkers.csv")
tees     = gpd.read_file("tees.csv")

# ------------------- CLEAN GEOMETRY -------------------
for df in [fairways, greens, bunkers, tees]:
    df = df[df.geometry.notna()]            # remove nulls
    df = df[~df.geometry.is_empty]          # remove empties

# ------------------- SET CRS IF MISSING -------------------
for df in [fairways, greens, bunkers, tees]:
    if df.crs is None:
        df.set_crs(epsg=4326, inplace=True)

# Convert everything to UTM Zone 18N (meters)
fairways = fairways.to_crs(32618)
greens   = greens.to_crs(32618)
bunkers  = bunkers.to_crs(32618)
tees     = tees.to_crs(32618)

# ------------------- 1) ASSIGN HOLE NUMBERS TO GREENS -------------------
greens["yc"] = greens.geometry.centroid.y
greens = greens.sort_values("yc", ascending=False).reset_index(drop=True)
greens["hole"] = greens.index + 1
greens.drop(columns=["yc"], inplace=True)


# ------------------- HELPER: SPATIAL NEAREST JOIN -------------------
def assign_nearest(target_gdf, ref_gdf):
    # spatial index nearest lookup
    idx = target_gdf.sindex.nearest(ref_gdf.geometry)
    # idx is tuple of (target_index, ref_index)
    matches = pd.DataFrame(idx[1], index=idx[0], columns=["nearest"])
    return target_gdf.join(ref_gdf["hole"].rename("hole").reindex(matches["nearest"]).set_axis(matches.index))


# ------------------- 2) ASSIGN HOLES TO FAIRWAYS (nearest green) -------------------
fairways = assign_nearest(fairways, greens)

# ------------------- 3) ASSIGN HOLES TO TEES (nearest fairway) -------------------
tees = assign_nearest(tees, fairways)

# ------------------- 4) ASSIGN HOLES TO BUNKERS (nearest fairway) -------------------
bunkers = assign_nearest(bunkers, fairways)

# ------------------- RESULTS CHECK -------------------
print("\n✅ Hole assignment complete.\n")
print("Greens per hole:", greens.hole.value_counts().sort_index().to_dict())
print("Fairways per hole:", fairways.hole.value_counts().sort_index().to_dict())
print("Tees per hole:", tees.hole.value_counts().sort_index().to_dict())
print("Bunkers per hole:", bunkers.hole.value_counts().sort_index().to_dict())



✅ Hole assignment complete.

Greens per hole: {1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 6: 1, 7: 1, 8: 1, 9: 1, 10: 1, 11: 1, 12: 1, 13: 1, 14: 1, 15: 1, 16: 1, 17: 1, 18: 1}
Fairways per hole: {1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 6: 1, 7: 1, 8: 3, 9: 2, 11: 1, 12: 1, 13: 1, 15: 1}
Tees per hole: {3.0: 1, 4.0: 1, 5.0: 1, 7.0: 1, 9.0: 2, 13.0: 2, 15.0: 1}
Bunkers per hole: {4.0: 1, 5.0: 1, 7.0: 1, 8.0: 1, 9.0: 1, 11.0: 1}


In [ ]:
# --- Upright hole plots for Clifton (UTM 18N) ---
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from shapely.affinity import rotate, translate
from shapely.geometry import Point

# ------------------- LOAD -------------------
fairways = gpd.read_file("fairways.csv")
greens   = gpd.read_file("greens.csv")
bunkers  = gpd.read_file("bunkers.csv")
tees     = gpd.read_file("tees.csv")

# ------------------- BASIC CLEANUP -------------------
def _clean(gdf):
    gdf = gdf[gdf.geometry.notna()].copy()
    if len(gdf):
        gdf = gdf[~gdf.geometry.is_empty]
    return gdf

fairways = _clean(fairways)
greens   = _clean(greens)
bunkers  = _clean(bunkers)
tees     = _clean(tees)

# Set CRS if missing, then project to meters (Clifton ~ UTM 18N)
for df in (fairways, greens, bunkers, tees):
    if df.crs is None:
        df.set_crs(epsg=4326, inplace=True)
    df.to_crs(32618, inplace=True)

# ------------------- HOLE NUMBERS -------------------
# Order greens by north->south to define hole numbers 1..N
greens["yc"] = greens.geometry.centroid.y
greens = greens.sort_values("yc", ascending=False).reset_index(drop=True)
greens["hole"] = greens.index + 1
greens.drop(columns="yc", inplace=True)

# Spatial index nearest assignment (robust & fast)
def assign_nearest(target_gdf, ref_gdf):
    # target_gdf gets a 'hole' column from nearest ref_gdf geometry
    ti, ri = target_gdf.sindex.nearest(ref_gdf.geometry)
    # Map target index -> ref index
    match = pd.Series(ri, index=ti)
    hole_map = ref_gdf["hole"]
    out = target_gdf.copy()
    out["hole"] = hole_map.reindex(match).set_axis(match.index)
    # If some features weren’t matched (rare), backfill with nearest by bounding-box fallback
    if out["hole"].isna().any():
        fallback_idx = out[out["hole"].isna()].index
        out.loc[fallback_idx, "hole"] = out.loc[fallback_idx].geometry.apply(
            lambda g: hole_map.loc[ref_gdf.distance(g).idxmin()]
        )
    return out

fairways = assign_nearest(fairways, greens)
tees     = assign_nearest(tees, fairways)   # tees near fairway of same hole
bunkers  = assign_nearest(bunkers, fairways)

# ------------------- PLOTTING UTILS -------------------
YARDS_PER_M = 1.0936133
M_PER_20Y = 20 / YARDS_PER_M   # gridline step in meters (~18.288 m)

def _rep_point(gdf):
    """Representative point for a set of polygons: centroid of unary_union."""
    if len(gdf) == 0:
        return None
    return gdf.unary_union.centroid

def _upright_geoms(gdfs, tee_pt, green_pt):
    """
    Rotate all geoms so vector (tee -> green) points straight up (positive Y),
    then translate so tee lies at (0,0).
    """
    if tee_pt is None or green_pt is None:
        return gdfs  # nothing to do
    dx = green_pt.x - tee_pt.x
    dy = green_pt.y - tee_pt.y
    angle_rad = np.arctan2(dy, dx)          # radians
    angle_deg = np.degrees(angle_rad)       # current heading
    rot_deg = 90 - angle_deg                # rotate so heading becomes 90° (up)

    out = []
    for gdf in gdfs:
        if gdf is None or len(gdf) == 0:
            out.append(gdf)
            continue
        g2 = gdf.copy()
        g2["geometry"] = g2.geometry.apply(lambda geom:
                                           translate(rotate(geom, rot_deg, origin=(tee_pt.x, tee_pt.y)),
                                                     xoff=-tee_pt.x, yoff=-tee_pt.y))
        out.append(g2)
    return out

def _hole_extent(fw_gdf, gr_gdf):
    """Tight but pleasant extent around a hole after rotation/translation."""
    # Heuristic width from fairway; fall back to greens if needed
    src = fw_gdf if (fw_gdf is not None and len(fw_gdf)) else gr_gdf
    if src is None or len(src) == 0:
        return (-50, 50, -10, 420 / YARDS_PER_M)  # default
    minx, miny, maxx, maxy = src.total_bounds
    width = max(60.0, (maxx - minx) * 1.25)     # at least ~60 m wide
    halfw = width / 2
    # Height: to the green centroid y plus a bit
    gcy = gr_gdf.unary_union.centroid.y if (gr_gdf is not None and len(gr_gdf)) else maxy
    ymax = max(gcy, maxy) * 1.05 + 15          # a little headroom
    return (-halfw, halfw, -10.0, ymax)

def _yards_formatter(values):
    return [f"{int(v * YARDS_PER_M):d}" for v in values]

def plot_hole_upright(h, save=True):
    fw  = fairways[fairways["hole"] == h]
    gr  = greens[greens["hole"] == h]
    ts  = tees[tees["hole"] == h]
    bk  = bunkers[bunkers["hole"] == h]

    if len(gr) == 0:
        print(f"Skipping hole {h}: no green geometry.")
        return

    # Choose representative tee: farthest tee centroid from green centroid (if many)
    g_pt = _rep_point(gr)
    if len(ts):
        t_centroids = ts.geometry.centroid
        far_idx = t_centroids.distance(g_pt).idxmax()
        t_pt = t_centroids.loc[far_idx]
    else:
        # If no tees captured, approximate tee as lowest fairway centroid
        if len(fw):
            f_pt = fw.unary_union.centroid
            t_pt = Point(f_pt.x, f_pt.y - 120)  # rough offset
        else:
            t_pt = None

    # Rotate + translate so tee->green points up and tee is at (0,0)
    fw_u, gr_u, bk_u, ts_u = _upright_geoms([fw, gr, bk, ts], t_pt, g_pt)

    # --- Figure ---
    fig, ax = plt.subplots(figsize=(4, 9), dpi=180)
    # Plot layers
    if len(fw_u): fw_u.plot(ax=ax, facecolor="#9FD18B", edgecolor="none")   # light green
    if len(bk_u): bk_u.plot(ax=ax, facecolor="#E5C59C", edgecolor="none")   # sand
    if len(gr_u): gr_u.plot(ax=ax, facecolor="#2F6B2F", edgecolor="none")   # dark green
    if len(ts_u): ts_u.plot(ax=ax, facecolor="#B10E1E", edgecolor="none")   # red tees

    # Extent
    xmin, xmax, ymin, ymax = _hole_extent(fw_u, gr_u)
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)

    # Gridlines every 20 yds (horizontal)
    step_m = M_PER_20Y
    y_ticks = np.arange(0, ymax + step_m, step_m)
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(_yards_formatter(y_ticks))
    ax.grid(axis="y", linestyle="--", linewidth=0.6, alpha=0.6)

    # Niceties
    ax.set_xlabel("")                       # no x label
    ax.set_ylabel("Yards from Tee")
    ax.set_title(f"Hole {h}", pad=10)
    ax.set_aspect("equal", adjustable="box")
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)

    if save:
        fname = f"hole_{h}_upright.png"
        print(f"Saved: {fname}")
        plt.close(fig)
    else:
        plt.show()

# ------------------- RUN OVER ALL HOLES -------------------
for h in sorted(greens["hole"].unique()):
    plot_hole_upright(h, save=True)


/var/folders/85/rzkn78t10p72g2_nfq5ytx3w0000gn/T/ipykernel_91186/1243881647.py:69: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  return gdf.unary_union.centroid
/var/folders/85/rzkn78t10p72g2_nfq5ytx3w0000gn/T/ipykernel_91186/1243881647.py:106: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  gcy = gr_gdf.unary_union.centroid.y if (gr_gdf is not None and len(gr_gdf)) else maxy
/var/folders/85/rzkn78t10p72g2_nfq5ytx3w0000gn/T/ipykernel_91186/1243881647.py:69: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  return gdf.unary_union.centroid
/var/folders/85/rzkn78t10p72g2_nfq5ytx3w0000gn/T/ipykernel_91186/1243881647.py:132: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  f_pt = fw.unary_union.centroid
/var/folders/85/rzkn78t10p72g2_nfq5ytx3w0000gn/T/ipykernel_91186/12438816

Saved: hole_1_upright.png
Saved: hole_2_upright.png
Saved: hole_3_upright.png


/var/folders/85/rzkn78t10p72g2_nfq5ytx3w0000gn/T/ipykernel_91186/1243881647.py:106: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  gcy = gr_gdf.unary_union.centroid.y if (gr_gdf is not None and len(gr_gdf)) else maxy
/var/folders/85/rzkn78t10p72g2_nfq5ytx3w0000gn/T/ipykernel_91186/1243881647.py:69: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  return gdf.unary_union.centroid
/var/folders/85/rzkn78t10p72g2_nfq5ytx3w0000gn/T/ipykernel_91186/1243881647.py:106: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  gcy = gr_gdf.unary_union.centroid.y if (gr_gdf is not None and len(gr_gdf)) else maxy
/var/folders/85/rzkn78t10p72g2_nfq5ytx3w0000gn/T/ipykernel_91186/1243881647.py:69: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  return gdf.unary_union.centroid


Saved: hole_4_upright.png
Saved: hole_5_upright.png
Saved: hole_6_upright.png
Saved: hole_7_upright.png


/var/folders/85/rzkn78t10p72g2_nfq5ytx3w0000gn/T/ipykernel_91186/1243881647.py:106: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  gcy = gr_gdf.unary_union.centroid.y if (gr_gdf is not None and len(gr_gdf)) else maxy
/var/folders/85/rzkn78t10p72g2_nfq5ytx3w0000gn/T/ipykernel_91186/1243881647.py:69: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  return gdf.unary_union.centroid
/var/folders/85/rzkn78t10p72g2_nfq5ytx3w0000gn/T/ipykernel_91186/1243881647.py:132: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  f_pt = fw.unary_union.centroid
/var/folders/85/rzkn78t10p72g2_nfq5ytx3w0000gn/T/ipykernel_91186/1243881647.py:106: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  gcy = gr_gdf.unary_union.centroid.y if (gr_gdf is not None and len(gr_gdf)) else maxy
/var/folders/85/rz

Saved: hole_8_upright.png
Saved: hole_9_upright.png


/var/folders/85/rzkn78t10p72g2_nfq5ytx3w0000gn/T/ipykernel_91186/1243881647.py:69: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  return gdf.unary_union.centroid
/var/folders/85/rzkn78t10p72g2_nfq5ytx3w0000gn/T/ipykernel_91186/1243881647.py:106: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  gcy = gr_gdf.unary_union.centroid.y if (gr_gdf is not None and len(gr_gdf)) else maxy
/var/folders/85/rzkn78t10p72g2_nfq5ytx3w0000gn/T/ipykernel_91186/1243881647.py:69: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  return gdf.unary_union.centroid
/var/folders/85/rzkn78t10p72g2_nfq5ytx3w0000gn/T/ipykernel_91186/1243881647.py:106: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  gcy = gr_gdf.unary_union.centroid.y if (gr_gdf is not None and len(gr_gdf)) else maxy
